# Notebook 09: Statistical Validation
Goal: Verify model stability and performance reliability


In [1]:
import pandas as pd
import numpy as np

# Load feature-engineered datasets
train_df = pd.read_csv("../notebooks_data/train_fe.csv")
test_df = pd.read_csv("../notebooks_data/test_fe.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (13130, 48)
Test shape: (3283, 48)


In [2]:
# Target column
TARGET_COLUMN = "class"

X_test = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN]

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Number of gesture classes:", y_test.nunique())


X_test shape: (3283, 47)
y_test shape: (3283,)
Number of gesture classes: 22


In [3]:
import joblib
from tensorflow.keras.models import load_model

# Load Random Forest (baseline)
rf_model = joblib.load(
    "../notebooks_data/models_data/advanced/random_forest.pkl"
)

# Load MLP (Keras model)
mlp_model = load_model(
    "../notebooks_data/models_data/baseline/mlp_baseline.hdf5"
)

print("Models loaded successfully")


Models loaded successfully


In [4]:
from sklearn.metrics import f1_score

# Random Forest predictions
rf_pred = rf_model.predict(X_test)
rf_f1 = f1_score(y_test, rf_pred, average="weighted")

# MLP predictions
mlp_probs = mlp_model.predict(X_test)
mlp_pred = np.argmax(mlp_probs, axis=1)
mlp_f1 = f1_score(y_test, mlp_pred, average="weighted")

print("Random Forest F1 Score:", round(rf_f1, 4))
print("MLP F1 Score:", round(mlp_f1, 4))


103/103 [==============================] - 0s 1ms/step
Random Forest F1 Score: 0.9921
MLP F1 Score: 0.9081


In [5]:
from scipy.stats import ttest_rel

# Simulated cross-validation scores (example for statistical testing)
rf_scores = np.random.normal(rf_f1, 0.01, 5)
mlp_scores = np.random.normal(mlp_f1, 0.01, 5)

t_stat, p_value = ttest_rel(mlp_scores, rf_scores)

print("T-statistic:", round(t_stat, 4))
print("P-value:", round(p_value, 4))


T-statistic: -13.4707
P-value: 0.0002


In [6]:
if p_value < 0.05:
    print("MLP performance improvement is statistically significant.")
else:
    print("No statistically significant difference detected.")


MLP performance improvement is statistically significant.


In [7]:
validation_summary = pd.DataFrame({
    "Model": ["Random Forest", "MLP"],
    "F1 Score": [rf_f1, mlp_f1],
    "Model Type": ["Tree-based", "Neural Network"]
})

validation_summary


,Model,F1 Score,Model Type
0,Random Forest,0.992085,Tree-based
1,MLP,0.908112,Neural Network
